In [ ]:
import pandas as pd
import numpy as np

# 示例数据
data = {
    'sub_id': [1, 2, 1, 12, 3, 5, 3, 4, 9, 10, 6],
    'parent_id': [None, None, None, None, 1, 2, 1, 1, 1, 2, 7]
}

df = pd.DataFrame(data)

# 去重
df = df.drop_duplicates()

# 所有帖子
posts = df[df['parent_id'].isna()]['sub_id'].drop_duplicates().to_frame('post_id')

# 所有评论
comments = df[df['parent_id'].notna()]

# 统计每个帖子的唯一评论数
comment_count = comments.groupby('parent_id')['sub_id'].nunique().reset_index()
comment_count.columns = ['post_id', 'number_of_comments']

# 左连接帖子，未评论的帖子补0
result = posts.merge(comment_count, on='post_id', how='left').fillna(0)
result['number_of_comments'] = result['number_of_comments'].astype(int)

# 按 post_id 排序
result = result.sort_values('post_id').reset_index(drop=True)

print(result)


In [ ]:
import pandas as pd

# 假设数据已读入 prices, units_sold
# prices: product_id, start_date, end_date, price
# units_sold : product_id, purchase_date, units

import pandas as pd

# 假设 prices, units_sold 已经加载
prices['start_date'] = pd.to_datetime(prices['start_date'])
prices['end_date'] = pd.to_datetime(prices['end_date'])

if not units_sold.empty:
    units_sold['purchase_date'] = pd.to_datetime(units_sold['purchase_date'])

# 1. 如果 UnitsSold 为空，直接让销量为 0
if units_sold.empty:
    result = prices[['product_id']].drop_duplicates()
    result['average_price'] = 0
    print(result)
else:
    # 2. 正常情况：先按 product_id merge
    df = units_sold.merge(prices, on='product_id', how='left')

    # 3. 过滤 purchase_date 落在价格区间
    df = df[
        (df['purchase_date'] >= df['start_date']) &
        (df['purchase_date'] <= df['end_date'])
    ]

    # 4. 计算金额
    df['amount'] = df['units'] * df['price']

    # 5. 聚合
    result = (
        df.groupby('product_id')
          .agg(total_amount=('amount','sum'),
               total_units=('units','sum'))
          .reset_index()
    )

    # 6. 合并以包含所有产品
    result = prices[['product_id']].drop_duplicates().merge(result, on='product_id', how='left')

    # 7. 计算平均售价
    result['average_price'] = (result['total_amount'] / result['total_units']).fillna(0)

    # 8. 四舍五入
    result['average_price'] = result['average_price'].round(2)

    print(result[['product_id', 'average_price']])

print(result)


In [ ]:
import pandas as pd

def page_recommendations(friendship, likes):

    # 1. 找到 user=1 的朋友（friendship 是双向的）
    friends = pd.concat([
        friendship.loc[friendship['user1_id'] == 1, 'user2_id'],
        friendship.loc[friendship['user2_id'] == 1, 'user1_id']
    ]).unique()

    # 2. 用户1 已喜欢的页面
    liked_by_user1 = likes.loc[likes['user_id'] == 1, 'page_id'].unique()

    # 3. 朋友喜欢的页面
    friend_pages = likes[likes['user_id'].isin(friends)]

    # 4. 去掉用户自己已喜欢的页面并去重
    recommended = (
        friend_pages.loc[~friend_pages['page_id'].isin(liked_by_user1), 'page_id']
        .drop_duplicates()
        .rename('recommended_page')
        .to_frame()
    )

    return recommended


In [ ]:
import pandas as pd

CEO = 1

# 第 1 层
level1 = employees.loc[(employees['manager_id'] == CEO) & 
                       (employees['employee_id'] != CEO), 'employee_id']

# 第 2 层
level2 = employees.loc[employees['manager_id'].isin(level1), 'employee_id']

# 第 3 层
level3 = employees.loc[employees['manager_id'].isin(level2), 'employee_id']

# 合并所有层级
result = pd.concat([level1, level2, level3]).drop_duplicates().sort_values()

print(result.to_frame('employee_id'))
